# Cumulative punishment analysis

In [ ]:
from scipy.io import loadmat
import pandas as pd
import os
import matplotlib.pyplot as plt
import numpy as np
from mapd import Trial, Table, Sinq
from mapd.kinematics import (
    detect_bouts_across_trials, rms_velocity,
    work, positive_effort, holding_cost,
    STATE_REST, STATE_DRIFT, STATE_MOVE,
    _STATE_COLORS, _STATE_LABEL,
)
from mapd.sinq_builders import build_composite_sinq
import h5py
from matplotlib.figure import Figure
from matplotlib.backends.backend_agg import FigureCanvasAgg as FigureCanvas
import matplotlib.colors as mcolors
import seaborn as sns

import plotly.express as px
from collections import defaultdict
import glob


import pickle
%load_ext autoreload
%autoreload 2
%matplotlib widget

def refresh_table_addons():
    import importlib
    import mapd.table_plotters as tps
    import mapd.table_scalars as tbs
    importlib.reload(tps)
    importlib.reload(tbs)

    for name in dir(tps):
        if name.startswith('_'):
            continue
        attr = getattr(tps, name)
        setattr(Table, name[len("plot_"):], attr)
    
    for name in dir(tbs):
        if name.startswith('_'):
            continue
        if name.startswith('compute_'):
            attr = getattr(tbs, name)
            setattr(Table, name[len("compute_"):], attr)

In [ ]:
# sinq2 = Sinq(sinqname='Fig2_learners')
# print(sinq2.__repr__())
# sinq2.df

sinq_0 = Sinq(sinqname='Figure1')
# sinq_1 = Sinq(sinqname='Fig1_control')
# sinq = build_composite_sinq(name='Figure2_learn_v_no_learn_lda',sources=[sinq_0,sinq_1])

# op_df = sinq.display_df(show_tags=True)
# op_df['learn'] = sinq.has_tag('learn')
# op_df['control'] = sinq.has_tag('control')
# op_df[['genotype','tags','learn','control']]

op_df = sinq_0.display_df(show_tags=True)
op_df['learn'] = sinq_0.has_tag('learn')
op_df['control'] = sinq_0.has_tag('control')
# op_df[['genotype','tags','learn','control']]

op_df = op_df.loc[op_df['learn']]
op_df[['genotype','tags','learn','control']]

In [ ]:
sinq2 = Sinq(sinqname='Fig2_learners')

In [ ]:
import matplotlib.patches as patches
import matplotlib as mpl
mpl.rcParams.update(mpl.rcParamsDefault)  # reset to defaults
mpl.rcParams['pdf.fonttype'] = 42         # embed fonts as text, not paths
mpl.rcParams['svg.fonttype'] = 'none'     # keep text editable in SVG
mpl.rcParams['font.family'] = 'Arial'
mpl.rcParams['font.size'] = 11

def cumulative_punishment(T:Table = None,cpdf = None, fig = None, ax = None, dfc = None, fig_dir:str = './Figure3',dur_v_trial_flag = False):
    output_directory = None

    if not T is None:
        fig = Figure(figsize=(6, 6), dpi=200)
        canvas = FigureCanvas(fig)
        ax = fig.add_subplot(111)

        cpdf = T.df[['pyasState','pyasXPosition','pyasWidth','probeZero','as_duration','on_target','time_on_target','op_cnd_blocks','total_duration']].copy()
        cpdf['cum_pun'] = cpdf['as_duration'].cumsum()
        cpdf['cum_dur'] = cpdf['total_duration'].cumsum()
        dfc = T.dfc
        output_directory = f'{fig_dir}/{T.dfc}'
        export_path = f'{fig_dir}/{T.dfc}/exports'

    rocket_cmap = sns.color_palette("rocket", as_cmap=True)

    cmin = 10
    cmax = -500

    _force_clrs = [
        (np.float64(0.95447591), np.float64(0.47082238), np.float64(0.32310953)),
        (np.float64(0.7965014), np.float64(0.10506637), np.float64(0.31063031)),
        ]
    if not dur_v_trial_flag:
        for ocb in cpdf['op_cnd_blocks'].unique():
            # print(ocb)
            T_rows = cpdf[cpdf['op_cnd_blocks']==ocb]
            
            tr_min = T_rows.index.min()
            tr_max = T_rows.index.max()
            width = tr_max-tr_min
            cum_block_min = T_rows.loc[tr_min,'cum_pun']
            cum_block_max = T_rows.loc[tr_max,'cum_pun']
            height = cum_block_max-cum_block_min

            # target patches
            tgt_clr = _force_clrs[0] if T_rows.loc[tr_min,'pyasState']=='lo' else _force_clrs[1]
            rect = patches.Rectangle(
                    (tr_min,cum_block_min),        # Bottom-left corner of the rectangle
                    width,           # Width (covers the specified rows)
                    height, # Height (covers all categories)    
                    linewidth=0,                        
                    edgecolor='none',
                    facecolor=tgt_clr,
                    alpha=.2
                )
            ax.add_patch(rect)

    y = cpdf['cum_pun']
    if dur_v_trial_flag:
        x = cpdf['cum_dur']/60
        x_label = 'Experiment duration (min)'
    else:
        x = cpdf.index
        x_label = 'Trial number'

    if not T is None:
        ax.plot(x,y,color='black',label=dfc)
    else:
        ax.plot(x,y,label=dfc)

    # ax.plot(T.df.index,T.df['on_target'].cumsum())
    # ax.plot(T.df.index,T.df['time_on_target'].cumsum())
    ax.set_xlabel(x_label)
    ax.set_ylabel("Cumulative punishment (s)")
    ax.set_title(f'Cumulative punishment')
    
    if not output_directory is None:    
        # fig.savefig(f'{output_directory}/cumulative_punishment{T.dfc}.svg',format='svg')
        fig.savefig(f'{output_directory}/cumulative_punishment{T.dfc}.png',format='png')
        cpdf.to_pickle(path=f'{export_path}/cumulative_punishment_{T.dfc}.pkl')

    return fig,ax



In [ ]:
sinq2.df

In [ ]:
T = sinq2.restore_table(dayflycell='210405_F1_C1')
# tr = Trial("D:\\Data\\210405\\210405_F1_C1\\LEDFlashWithPiezoCueControl_Raw_210405_F1_C1_13.mat")

# import h5py
# h5py.File("D:\\Data\\210405\\210405_F1_C1\\LEDFlashWithPiezoCueControl_Raw_210405_F1_C1_12.mat", 'r',libver='latest', swmr=True)

In [ ]:
T.extract_trial_properties()
T.df.loc[185,'as_duration']
T.extract_trial_properties(prop_list = ['on_target'])
T.extract_trial_properties(prop_list = ['time_on_target'])
fig,ax = cumulative_punishment(T)
display(fig)

In [ ]:
T.df['as_duration'].head(50)

In [ ]:
for dfc in sinq2.df.index:
    T = sinq2.restore_table(dayflycell=dfc)
    T.extract_trial_properties()
    T.extract_trial_properties(prop_list = ['on_target'])
    T.extract_trial_properties(prop_list = ['time_on_target'])

    fig,ax = cumulative_punishment(T)
    sinq2.drop_tables()

In [ ]:
# Now load and plot the plots together
cpdf = defaultdict(dict)

ftrfig = Figure(figsize=(12,12))
# plt.figure(figsize=(5,5))
ftrax = ftrfig.add_subplot(1, 1, 1)
dur_v_trial_flag = False

for dfc in sinq2.df.index:
    export_path = f'./Figure2/{dfc}/exports'
    pklpat = f'{export_path}/cumulative_punishment_{dfc}.pkl'
    file_path = glob.glob(pklpat)
    if file_path:
        file_path = file_path[0]
        print(file_path)

    cpdf[dfc] = pd.read_pickle(file_path)
    # cpdf[dfc]['cum_dur'] = cpdf[dfc]['total_duration'].cumsum()
    ftrfig,ftrax = cumulative_punishment(cpdf=cpdf[dfc],fig = ftrfig,ax=ftrax,dur_v_trial_flag=dur_v_trial_flag,dfc=dfc)

if not dur_v_trial_flag:
    x = [0,  400]
    y = [0, 400*10]
    ftrax.plot(x,y,ls='--',color='gray')

    x = [0,  1000]
    y = [0, 1000*1]
    ftrax.plot(x,y,ls='--',color='gray')

ftrax.legend()
display(ftrfig)
ftrfig.savefig('./Figure2/cumulative_punishment_over_trials_all_flies_trials.svg')


## Select a couple punishment schedules to replay to new flies

In [ ]:
# for dfc in sinq2.df.index:
trofinterestdict = { # '210302_F1_C2':[40],  # big step early on
                    # '250304_F3_C1':[[700, 1270, 825]],   # too good
                    '250924_F2_C1':[40], # consistent punishment
                    '250228_F2_C1':[130]
                    }

for dfc in trofinterestdict.keys():  #'210302_F1_C2'
    T = sinq2.restore_table(dayflycell=dfc)
    T.extract_trial_properties()
    T.extract_trial_properties(prop_list = ['on_target'])
    T.extract_trial_properties(prop_list = ['time_on_target'])

    fig,ax = cumulative_punishment(T,dur_v_trial_flag=False)
    display(fig)

    tr_0 = trofinterestdict[dfc]
    ftrfig = Figure(figsize=(12,4))
    ftrax = ftrfig.add_subplot(1, 1, 1)
    for tr in tr_0:
        ftrax = T.plot_some_probe_groups(index=T.df.loc[tr:tr+30].index,ax=ftrax,savefig=False,from_zero=True,format='svg')

        display(ftrfig)
        ftrfig.savefig(f'./Figure2/{T.dfc}/cumulative_punishment_{T.dfc}_{tr}_{tr+30}.png')

    sinq2.drop_tables()

In [ ]:
cpdf = pd.read_pickle('./Figure2/250924_F2_C1/exports/cumulative_punishment_210302_F1_C2.pkl')
cpdf

In [ ]:
# In Python
import pandas as pd
import scipy.io as sio
import pickle


pkls = [
        './Figure2/250228_F2_C1/exports/cumulative_punishment_250228_F2_C1.pkl',
        './Figure2/250924_F2_C1/exports/cumulative_punishment_250924_F2_C1.pkl',
        ]

for pk in pkls:
    with open(pk, 'rb') as f:
        data = pickle.load(f)

    # If it's a DataFrame
    if isinstance(data, pd.DataFrame):
        sio.savemat(pk.replace('.pkl', '.mat'), {'data': data.to_dict('list')})
    # or just pickle -> mat for general dict-like objects
    else:
        sio.savemat(pk.replace('.pkl', '.mat'), {'data': data})